In [1]:
import os
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import json

## Load Data

In [2]:
filtered_data_path = f"{os.environ['HOME']}/.cz-benchmarks/datasets/replogle_k562_essential_perturbpredict_de_results_control_cells_v1.h5ad"
# filtered_data_path = "/data2/czbenchmarks/replogle2022/K562_essential_raw_singlecell_01.h5ad"

adata_filtered = ad.read_h5ad(filtered_data_path, backed=None)
adata_filtered.obs = adata_filtered.obs.rename(columns={'gene_id':'condition', 'gene':'condition_name'})
adata_filtered.var = adata_filtered.var.rename(columns={'gene_name':'gene'})
adata_filtered.obs['condition'] = adata_filtered.obs['condition'].astype(str)

control_cells_ids_cr4_orig = adata_filtered.uns["control_cells_ids"]
control_cells_ids_cr4_orig.pop("non-targeting", None)

array([], dtype=float64)

In [3]:
cr4_data_path = (
    "/data2/czbenchmarks/replogle2022/raw_h5ad_from_cr4/K562_essential_mtx.h5ad"
)
adata_cr4 = ad.read_h5ad(cr4_data_path, backed=None)
adata_cr4.var.index.name = "gene_id"
adata_cr4.var.rename(columns={"gene_name": "gene"}, inplace=True)
adata_cr4.obs.rename(columns={"gene_id": "condition"}, inplace=True)

In [4]:
with open("/data2/czbenchmarks/control_cells_ids_replogle_k562_essential_perturbpredict.json", "r") as f:
    control_cells_ids_filtered = json.load(f) # filtered data

with open("/data2/czbenchmarks/control_cells_ids_replogle_k562_essential_perturbpredict_validate.json", "r") as f:
    control_cells_ids_cr4_validate = json.load(f) # cr4 data

## QC of Conditions

In [5]:
len(control_cells_ids_cr4_orig.keys()), len(control_cells_ids_filtered.keys()), len(control_cells_ids_cr4_validate.keys())

(2057, 2057, 2057)

In [6]:
set(control_cells_ids_filtered.keys()) == set(control_cells_ids_cr4_orig.keys()) == set(control_cells_ids_cr4_validate.keys())

True

In [7]:
set(control_cells_ids_filtered.keys()).issubset(set(control_cells_ids_cr4_orig.keys()))

True

In [8]:
set(control_cells_ids_cr4_validate.keys()).issubset(set(control_cells_ids_cr4_orig.keys()))

True

## QC of Matches

In [11]:
# conditions = list(control_cells_ids_filtered.keys())
conditions = list(control_cells_ids_cr4_orig.keys())

# Old and new implementation fairly similar when both run on same dataset (cr4)
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_orig[condition]), len(control_cells_ids_cr4_validate[condition]))
    assert len(control_cells_ids_cr4_orig[condition]) == len(control_cells_ids_cr4_validate[condition])
    
    matches = 0
    for pos, (val1, val2) in enumerate(zip(control_cells_ids_cr4_orig[condition], control_cells_ids_cr4_validate[condition].values())):
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {len(control_cells_ids_cr4_orig[condition]) - matches}")

ENSG00000001497 83 83
     matches 83, differences 0
ENSG00000003509 18 18
     matches 18, differences 0
ENSG00000004779 160 160
     matches 159, differences 1
ENSG00000004897 366 366
     matches 364, differences 2
ENSG00000005007 131 131
     matches 131, differences 0
ENSG00000005100 140 140
     matches 140, differences 0
ENSG00000005175 18 18
     matches 18, differences 0
ENSG00000005194 151 151
     matches 151, differences 0
ENSG00000005448 208 208
     matches 206, differences 2
ENSG00000006634 131 131
     matches 130, differences 1
ENSG00000006695 94 94
     matches 94, differences 0
ENSG00000006712 238 238
     matches 235, differences 3
ENSG00000006715 332 332
     matches 331, differences 1
ENSG00000006744 85 85
     matches 85, differences 0
ENSG00000007168 126 126
     matches 123, differences 3
ENSG00000007866 267 267
     matches 265, differences 2
ENSG00000007923 46 46
     matches 46, differences 0
ENSG00000008018 143 143
     matches 141, differences 2
ENSG000000

In [12]:
# Results are very different between old cr4 version and new filtered version
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_orig[condition]), len(control_cells_ids_filtered[condition]))
    assert len(control_cells_ids_cr4_orig[condition]) == len(control_cells_ids_filtered[condition])
    
    matches = 0
    for pos, (val1, val2) in enumerate(zip(control_cells_ids_cr4_orig[condition], control_cells_ids_filtered[condition].values())):
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {len(control_cells_ids_cr4_orig[condition]) - matches}")

ENSG00000001497 83 83
     matches 1, differences 82
ENSG00000003509 18 18
     matches 0, differences 18
ENSG00000004779 160 160
     matches 0, differences 160
ENSG00000004897 366 366
     matches 4, differences 362
ENSG00000005007 131 131
     matches 0, differences 131
ENSG00000005100 140 140
     matches 1, differences 139
ENSG00000005175 18 18
     matches 1, differences 17
ENSG00000005194 151 151
     matches 1, differences 150
ENSG00000005448 208 208
     matches 1, differences 207
ENSG00000006634 131 131
     matches 1, differences 130
ENSG00000006695 94 94
     matches 0, differences 94
ENSG00000006712 238 238
     matches 1, differences 237
ENSG00000006715 332 332
     matches 1, differences 331
ENSG00000006744 85 85
     matches 1, differences 84
ENSG00000007168 126 126
     matches 1, differences 125
ENSG00000007866 267 267
     matches 1, differences 266
ENSG00000007923 46 46
     matches 1, differences 45
ENSG00000008018 143 143
     matches 1, differences 142
ENSG000000

In [23]:
# But the differences appear to be mostly because of sort order. There are still some unexplained issues with the original dictionary because differences of 1 can't be explained by sorting
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_validate[condition]), len(control_cells_ids_filtered[condition]))
    assert len(control_cells_ids_cr4_validate[condition]) == len(control_cells_ids_filtered[condition])
    
    matches = 0
    num_pairs = len(control_cells_ids_cr4_validate[condition])
    for pos, key in enumerate(control_cells_ids_cr4_validate[condition].keys()):
        val1, val2 = control_cells_ids_cr4_validate[condition][key], control_cells_ids_filtered[condition][key]
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {num_pairs - matches}")

ENSG00000001497 83 83
     matches 83, differences 0
ENSG00000003509 18 18
     matches 18, differences 0
ENSG00000004779 160 160
     matches 160, differences 0
ENSG00000004897 366 366
     matches 366, differences 0
ENSG00000005007 131 131
     matches 131, differences 0
ENSG00000005100 140 140
     matches 140, differences 0
ENSG00000005175 18 18
     matches 18, differences 0
ENSG00000005194 151 151
     matches 151, differences 0
ENSG00000005448 208 208
     matches 208, differences 0
ENSG00000006634 131 131
     matches 131, differences 0
ENSG00000006695 94 94
     matches 94, differences 0
ENSG00000006712 238 238
     matches 238, differences 0
ENSG00000006715 332 332
     matches 332, differences 0
ENSG00000006744 85 85
     matches 85, differences 0
ENSG00000007168 126 126
     matches 126, differences 0
ENSG00000007866 267 267
     matches 267, differences 0
ENSG00000007923 46 46
     matches 46, differences 0
ENSG00000008018 143 143
     matches 143, differences 0
ENSG000000

## Unused Cells

In [13]:
control_cells_used = set()
for condition in conditions:
    control_cells_used = control_cells_used.union(set(control_cells_ids_filtered[condition].keys()))
    control_cells_used = control_cells_used.union(set(control_cells_ids_filtered[condition].values()))
    

In [14]:
all_cells = set(adata_filtered.obs_names)

In [15]:
len(all_cells), len(control_cells_used)

(310385, 310281)

In [16]:
control_cells_used.issubset(all_cells)

True

In [17]:
unused_cell_ids = all_cells.difference(control_cells_used)

In [18]:
unused_cells = adata_filtered.obs.loc[list(unused_cell_ids)]

In [19]:
unused_cells.condition.astype(str).unique()

array(['non-targeting'], dtype=object)

In [20]:
unused_cells

,gem_group,condition_name,condition,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count
cell_barcode,,,,,,,,,,,
TCCACGTGTCATCACA-26,26,non-targeting,non-targeting,non-targeting,11179_non-targeting_non-targeting_non-targeting,non-targeting_02787|non-targeting_01465,0.098578,18777.0,1.571038,0.901381,20831.373047
GGGACTCTCACCTCTG-25,25,non-targeting,non-targeting,non-targeting,10772_non-targeting_non-targeting_non-targeting,non-targeting_00166|non-targeting_02207,0.119748,12877.0,0.203468,0.855594,15050.363281
CCATAAGTCGACGACC-6,6,non-targeting,non-targeting,non-targeting,10822_non-targeting_non-targeting_non-targeting,non-targeting_00442|non-targeting_01312,0.078876,12995.0,-0.127853,0.954127,13619.778320
CCACGAGCATTCTCTA-19,19,non-targeting,non-targeting,non-targeting,10884_non-targeting_non-targeting_non-targeting,non-targeting_00807|non-targeting_02911,0.080580,17523.0,0.372891,1.123384,15598.401367
TTGAACGAGAGTATAC-8,8,non-targeting,non-targeting,non-targeting,10859_non-targeting_non-targeting_non-targeting,non-targeting_00652|non-targeting_02540,0.086616,18634.0,1.354366,0.950474,19604.958984
...,...,...,...,...,...,...,...,...,...,...,...
AATTTCCTCCGAAGGA-12,12,non-targeting,non-targeting,non-targeting,10882_non-targeting_non-targeting_non-targeting,non-targeting_00799|non-targeting_00833,0.093719,13978.0,-0.502728,1.232812,11338.302734
TACCGGGCATGTTCGA-34,34,non-targeting,non-targeting,non-targeting,10872_non-targeting_non-targeting_non-targeting,non-targeting_00702|non-targeting_02833,0.106195,15029.0,-0.047092,1.091537,13768.660156
CATCGTCTCGTTACCC-12,12,non-targeting,non-targeting,non-targeting,11023_non-targeting_non-targeting_non-targeting,non-targeting_01798|non-targeting_01119,0.114911,15525.0,-0.266281,1.232812,12593.157227


## Prototype Algorithm

In [32]:
n_nonzero_genes = (adata_filtered.X > 0).sum(axis=1)
if not isinstance(n_nonzero_genes, np.ndarray):
    n_nonzero_genes = n_nonzero_genes.toarray()

adata_filtered.obs['n_nonzero_genes'] = n_nonzero_genes
assert (adata_filtered.obs['UMI_count'] >= adata_filtered.obs['n_nonzero_genes']).all()

adata_filtered.obs[['UMI_count', 'n_nonzero_genes']].head()

,UMI_count,n_nonzero_genes
cell_barcode,,
AAACCCAAGAAATCCA-27,11438.0,3332
AAACCCAAGAACTTCC-31,5342.0,2192
AAACCCAAGAAGCCAC-34,17305.0,4002
AAACCCAAGAATAGTC-43,30244.0,5358
AAACCCAAGACAGCGT-28,8407.0,2944


In [28]:
assert (adata_filtered.obs['UMI_count'] >= adata_filtered.obs['n_nonzero_genes']).all()

In [30]:
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.neighbors import NearestNeighbors

In [36]:
obs.gem_group.unique()

array([27, 31, 34, 43, 28, 10, 19, 11, 32, 24, 42, 40, 44, 38, 12,  7, 20,
       14,  5, 22, 21, 23, 47, 18, 17,  1, 45, 25, 16, 33, 26, 39,  6,  9,
        2, 30,  3, 48,  8, 29, 46,  4, 36, 15, 41, 37, 35, 13])

In [34]:
obs = adata_filtered.obs
# ['UMI_count', 'n_nonzero_genes', 'gem_group', 'condition']

,UMI_count,n_nonzero_genes,gem_group,condition
cell_barcode,,,,
AAACCCAAGAAATCCA-27,11438.0,3332,27,ENSG00000145414
AAACCCAAGAACTTCC-31,5342.0,2192,31,ENSG00000169679
AAACCCAAGAAGCCAC-34,17305.0,4002,34,ENSG00000198258
AAACCCAAGAATAGTC-43,30244.0,5358,43,ENSG00000171159
AAACCCAAGACAGCGT-28,8407.0,2944,28,ENSG00000100575


In [49]:
# n_nonzero_genes = (adata_filtered.X > 0).sum(axis=1)
# if not isinstance(n_nonzero_genes, np.ndarray):
#     n_nonzero_genes = n_nonzero_genes.toarray()

# adata_filtered.obs['n_nonzero_genes'] = n_nonzero_genes
# assert (adata_filtered.obs['UMI_count'] >= adata_filtered.obs['n_nonzero_genes']).all()

obs = adata_filtered.obs
    
condition_key = 'condition'
control_condition = "non-targeting"
condition_list = [x for x in obs[condition_key].unique() if x != control_condition]

grouper_key = 'gem_group'

control_mask = obs.condition == control_condition
control_cells = obs.loc[control_mask]
condition_cells = obs.loc[~control_mask]

condition = conditions[0]

gem_group = 4
control_ = control_cells.loc[control_cells[grouper_key] == gem_group, ['UMI_count', 'n_nonzero_genes']]
condition_ = condition_cells.loc[(condition_cells[condition_key] == condition) & (condition_cells[grouper_key] == gem_group), 
    ['UMI_count', 'n_nonzero_genes']]

print(control_.shape, condition_.shape)

(120, 2) (4, 2)


In [72]:
dist_matrix = pairwise_distances(condition_, control_)
dist_matrix.shape

(4, 120)

In [73]:
dist_matrix.argmin(axis=1)

array([ 83, 109,  57,  31])

In [74]:
dist_matrix.argsort(axis=1)[:, :10]

array([[ 83, 104,  61, 100,  90,  30,   1,  88,  95,  75],
       [109,   3,  75,  76,  95,  88,  57,   1,  90, 100],
       [ 57,  71,  27, 117,   3,  76,  29, 109,  46,  75],
       [ 31,   5,  58, 114,  93,  52,  30,  61,  83,   9]])

In [80]:
dist_matrix[:, dist_matrix.argmin(axis=1)]

array([[  39.2173431 , 1080.82375992, 1911.94168321, 2280.66591153],
       [1293.00618715,  240.01041644,  657.00076103, 3526.25410315],
       [2138.18334106, 1031.6559504 ,  195.2562419 , 4360.83719027],
       [1871.1988136 , 2959.44065661, 3807.16640036,  370.74249824]])

In [75]:

rows, cols = linear_sum_assignment(dist_matrix)

In [76]:
rows

array([0, 1, 2, 3])

In [64]:
cols

array([ 83, 109,  57,  31])

In [81]:
dist_matrix[rows, cols]

array([ 39.2173431 , 240.01041644, 195.2562419 , 370.74249824])

In [100]:
columns

['condition', 'gem_group', ['UMI_count', 'n_nonzero_genes']]

In [117]:
from collections import defaultdict
from scipy.optimize import linear_sum_assignment

# # Calculate number of non-zero genes
# n_nonzero_genes = (adata_filtered.X > 0).sum(axis=1)
# if not isinstance(n_nonzero_genes, np.ndarray):
#     n_nonzero_genes = n_nonzero_genes.toarray()

# adata_filtered.obs['n_nonzero_genes'] = n_nonzero_genes
# assert (adata_filtered.obs['UMI_count'] >= adata_filtered.obs['n_nonzero_genes']).all()

condition_key = 'condition'
group_key = 'gem_group'
dist_keys = ['UMI_count', 'n_nonzero_genes'] 
columns = [condition_key, group_key] + dist_keys
control_condition = "non-targeting"

obs = adata_filtered.obs[columns]

control_mask = obs.condition == control_condition
control_cells = obs.loc[control_mask]
treatment_cells = obs.loc[~control_mask]

control_cells_gem = {gem_group:df for gem_group,df in control_cells.groupby(group_key)}
control_cells_ids = defaultdict(dict)

for condition, treatment_cells_cond in treatment_cells.groupby(condition_key):
    for gem_group, treatment_cells_cond_group in treatment_cells_cond.groupby(group_key):
        control_ = control_cells_gem[gem_group][dist_keys]
        treatment_ = treatment_cells_cond_group[dist_keys]
        # print(f"{condition} {gem_group} {len(treatment_)} {len(control_)}")
        
        dist_matrix = pairwise_distances(treatment_, control_)
        treatment_indices, control_indices = linear_sum_assignment(dist_matrix)

        assert np.allclose(control_indices[treatment_indices], dist_matrix.argmin(axis=1))
        
        treatment_ids = treatment_.index[treatment_indices]
        control_ids = control_.index[control_indices]
        control_cells_ids[condition].update(dict(zip(treatment_ids, control_ids)))

# Can probably accelerate by inverting loop order
# Will run fewer pairwise distance calculations, albeit with memory increase
for gem_group, treatment_gem_group in treatment_cells.groupby(group_key):
    control_ = control_cells_gem[gem_group][dist_keys]
    treatment_ = treatment_gem_group[dist_keys]
    dist_matrix = pairwise_distances(treatment_, control_)
    
    for condition, treatment_gem_condition in treatment_gem_group.groupby(condition_key):
        treatment_indices, control_indices = linear_sum_assignment(dist_matrix[treatment_gem_condition.indexes])

        treatment_ids = treatment_.index[treatment_indices]
        control_ids = control_.index[control_indices]
        control_cells_ids[condition].update(dict(zip(treatment_ids, control_ids)))

AssertionError: 

In [118]:
control_indices[treatment_indices]

array([156, 101, 106,  12, 138])

In [119]:
dist_matrix.argmin(axis=1)

array([156, 101, 106,  12, 156])

In [120]:
condition, gem_group

('ENSG00000001497', 39)

In [123]:
dist_matrix[treatment_indices, control_indices]

array([13.92838828, 75.42545989, 39.01281841, 84.8763807 , 92.84934033])

In [124]:
dist_matrix[treatment_indices, dist_matrix.argmin(axis=1)]

array([13.92838828, 75.42545989, 39.01281841, 84.8763807 , 82.29216245])

In [109]:
control_cells_ids['ENSG00000001497']

{'CCTCTCCAGCGTATGG-48': 'GTTGAACAGCTGACAG-48'}

In [110]:
control_cells_ids = defaultdict(dict)
control_cells_ids['b'].update({'a':5})
control_cells_ids['b'].update({'c':7})
control_cells_ids

defaultdict(dict, {'b': {'a': 5, 'c': 7}})

In [114]:
control_indices[treatment_indices]

array([ 74,  25, 167, 140])

In [115]:
dist_matrix.argmin(axis=1)

array([ 74,  25, 167, 140])

In [116]:
treatment_indices

array([0, 1, 2, 3])